# Scheduling prices Gold

Prepare the validated hourly spot-price dataset for a historical equipment scheduler.

Norwegian local dates from 2024-01-01 through 2025-12-31. The data is historical, not a forecast.

## Project paths and Spark

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SILVER_ROOT = PROJECT_ROOT / "data" / "silver"
GOLD_ROOT = PROJECT_ROOT / "data" / "gold"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.appName("DAT535 Gold Scheduling")
    .master("local[4]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark timezone:", spark.conf.get("spark.sql.session.timeZone"))

26/11/21 15:05:25 WARN Utils: Your hostname, homelab resolves to a loopback address: 127.0.1.1; using 192.168.10.53 instead (on interface lan0)
26/11/21 15:05:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/21 15:05:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/21 15:05:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark timezone: UTC


## Make Gold prices
Uses Silver spot-price output

In [2]:
from gold_layer.scheduling import ingest

result = ingest(
    silver_root=SILVER_ROOT,
    gold_root=GOLD_ROOT,
    start="2024-01-01",
    end="2026-01-01",
    spark=spark,
)

print(result)

{'dataset': 'scheduling_prices', 'status': 'success', 'rows': 87374, 'gold_path': '/home/roland/UiS/rewrite/group14/data/gold/scheduling/prices'}


## 3. Inspect Gold data

In [3]:
prices = spark.read.parquet(str(GOLD_ROOT / "scheduling" / "prices"))
prices.printSchema()
prices.orderBy("price_area", "start_time").show(10, truncate=False)

root
 |-- price_area: string (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- nok_per_kwh: decimal(18,6) (nullable = true)

+----------+-------------------+-------------------+-----------+
|price_area|start_time         |end_time           |nok_per_kwh|
+----------+-------------------+-------------------+-----------+
|NO1       |2023-12-31 23:00:00|2024-01-01 00:00:00|0.604630   |
|NO1       |2024-01-01 00:00:00|2024-01-01 01:00:00|0.550560   |
|NO1       |2024-01-01 01:00:00|2024-01-01 02:00:00|0.308550   |
|NO1       |2024-01-01 02:00:00|2024-01-01 03:00:00|0.275170   |
|NO1       |2024-01-01 03:00:00|2024-01-01 04:00:00|0.269880   |
|NO1       |2024-01-01 04:00:00|2024-01-01 05:00:00|0.238640   |
|NO1       |2024-01-01 05:00:00|2024-01-01 06:00:00|0.254260   |
|NO1       |2024-01-01 06:00:00|2024-01-01 07:00:00|0.281460   |
|NO1       |2024-01-01 07:00:00|2024-01-01 08:00:00|0.294950   |
|NO1       |2024-01-01 08:00:00|20

## Quality check Gold data

In [4]:
total = prices.count()

duplicates = (
    prices.groupBy("price_area", "start_time")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_duration = prices.filter(
    F.col("end_time").cast("long") - F.col("start_time").cast("long") != 3600
).count()

nulls = prices.filter(
    F.col("price_area").isNull()
    | F.col("start_time").isNull()
    | F.col("end_time").isNull()
    | F.col("nok_per_kwh").isNull()
).count()

print("Gold rows:", total)
print("Duplicate keys:", duplicates)
print("Non-hourly intervals:", invalid_duration)
print("Null required fields:", nulls)

assert total > 0
assert duplicates == 0
assert invalid_duration == 0
assert nulls == 0

Gold rows: 87374
Duplicate keys: 0
Non-hourly intervals: 0
Null required fields: 0


## Price areas and coverage
Compare available hourly slots with the expected hours. Coverage below 100% is expected because some DST Bronze batches were quarantined in Silver.

In [5]:
from datetime import UTC, date, datetime, time
from zoneinfo import ZoneInfo

oslo = ZoneInfo("Europe/Oslo")
first = datetime.combine(date(2024, 1, 1), time.min, oslo).astimezone(UTC)
last = datetime.combine(date(2026, 1, 1), time.min, oslo).astimezone(UTC)
expected_hours = int((last - first).total_seconds() / 3600)

counts = {
    row["price_area"]: row["count"]
    for row in prices.groupBy("price_area").count().collect()
}

print("Expected hourly slots per area:", expected_hours)
for area in ("NO1", "NO2", "NO3", "NO4", "NO5"):
    actual = counts.get(area, 0)
    print(f"{area}: {actual:,} / {expected_hours:,} ({actual / expected_hours:.2%})")

Expected hourly slots per area: 17544
NO1: 17,470 / 17,544 (99.58%)
NO2: 17,494 / 17,544 (99.72%)
NO3: 17,470 / 17,544 (99.58%)
NO4: 17,470 / 17,544 (99.58%)
NO5: 17,470 / 17,544 (99.58%)


## 6. Find gaps
A gap is a break between consecutive one-hour intervals.

In [6]:
from pyspark.sql import Window

order = Window.partitionBy("price_area").orderBy("start_time")
gaps = (
    prices.withColumn("previous_end", F.lag("end_time").over(order))
    .filter(
        F.col("previous_end").isNotNull()
        & (F.col("start_time") != F.col("previous_end"))
    )
    .select("price_area", "previous_end", "start_time")
)
print("Discontinuities:", gaps.count())
gaps.orderBy("price_area", "previous_end").show(20, truncate=False)

Discontinuities: 10
+----------+-------------------+-------------------+
|price_area|previous_end       |start_time         |
+----------+-------------------+-------------------+
|NO1       |2024-10-26 22:00:00|2024-10-28 23:00:00|
|NO1       |2025-10-25 22:00:00|2025-10-26 23:00:00|
|NO2       |2024-10-26 22:00:00|2024-10-27 23:00:00|
|NO2       |2025-10-25 22:00:00|2025-10-26 23:00:00|
|NO3       |2024-10-26 22:00:00|2024-10-28 23:00:00|
|NO3       |2025-10-25 22:00:00|2025-10-26 23:00:00|
|NO4       |2024-10-26 22:00:00|2024-10-28 23:00:00|
|NO4       |2025-10-25 22:00:00|2025-10-26 23:00:00|
|NO5       |2024-10-26 22:00:00|2024-10-28 23:00:00|
|NO5       |2025-10-25 22:00:00|2025-10-26 23:00:00|
+----------+-------------------+-------------------+

